# Agricultural Data Cleaning: USDA NASS Agriculture Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [ ]:
data_path = r'../../../../data/tabular/agricultural/raw'
df_ag = pd.read_csv(f'{data_path}/usdaNass-agriculture.csv')

In [ ]:
# Step 1: Inspect
print(df_ag.shape)
df_ag.head()

In [ ]:
# Step 2: Rename columns - lowercase, replace spaces with underscores
df_ag.columns = df_ag.columns.str.strip().str.lower().str.replace(" ", "_")

# Step 3: Drop columns with >90% missing values
df_ag.replace(r'^\s*$', pd.NA, regex=True, inplace=True)
threshold = len(df_ag) * 0.10
df_ag.dropna(axis=1, thresh=threshold, inplace=True)

print("Remaining columns:", df_ag.columns.tolist())

In [ ]:
# Step 4: Fix Value column
# USDA suppresses some values with special codes - replace with NaN
suppressed = {'(D)', '(H)', '(Z)', '(NA)', ' (D)', ' (H)', ' (Z)', ' (NA)'}
df_ag['value'] = df_ag['value'].apply(lambda x: np.nan if str(x).strip() in suppressed else x)
df_ag['value'] = df_ag['value'].astype(str).str.replace(',', '', regex=False)
df_ag['value'] = pd.to_numeric(df_ag['value'], errors='coerce')

# Fix CV (%) column
df_ag['cv_(%)'] = df_ag['cv_(%)'].apply(lambda x: np.nan if str(x).strip() in suppressed else x)
df_ag['cv_(%)'] = pd.to_numeric(df_ag['cv_(%)'], errors='coerce')

print(f"Missing in 'value': {df_ag['value'].isnull().sum()}")
df_ag.dtypes

In [ ]:
# Step 5: Save Cleaned Data
import os
out_path = '../../../../data/tabular/agricultural/cleaned'
os.makedirs(out_path, exist_ok=True)
df_ag.to_csv(f'{out_path}/usdaNass-agriculture-clean.csv', index=False)
print(f"Saved {len(df_ag):,} rows → {out_path}/usdaNass-agriculture-clean.csv")